In [1]:
import json
import os
import time
import faiss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig
from sentence_transformers import SentenceTransformer
import asyncio
from google import genai
from google.genai import types

os.environ["DISABLE_TQDM"] = "1"

Load Documents

In [2]:
DATASET_PATH = "../Dataset"

def load_documents():
    docs = {}
    doc_dir = os.path.join(DATASET_PATH, "documents")
    
    for file in os.listdir(doc_dir):
        if file.endswith(".txt"):
            name = file.replace(".txt", "")
            file_path = os.path.join(doc_dir, file)
            with open(file_path, "r", encoding="utf-8") as f:
                docs[name] = f.read()
    return docs

def load_all_qa():
    all_qa = []
    qa_dir = os.path.join(DATASET_PATH, "qa")
    
    for file in os.listdir(qa_dir):
        if file.endswith(".json"):
            file_path = os.path.join(qa_dir, file)
            with open(file_path, "r", encoding="utf-8") as f:
                all_qa.extend(json.load(f))
    return all_qa

Load LLM models

In [3]:
# Project configurations
PROJECT_ID = "gen-lang-client-0279093715"
LOCATION = "us-central1"
MODEL_ID = "gemini-2.5-flash"

# Initialize the Gen AI Client for Vertex AI
client = genai.Client(
    vertexai=True, 
    project=PROJECT_ID, 
    location=LOCATION
)

# Semaphore to control concurrency
rate_limiter = asyncio.Semaphore(1)

Metric Utils

In [4]:
# Token Counter
def count_tokens(tokenizer, text):
    return len(tokenizer(text)["input_ids"])

In [5]:
# Exact Match
def exact_match(pred, gold):
    return int(pred.strip() == gold.strip())

In [6]:
async def generate_async(prompt, model_id=MODEL_ID):
    async with rate_limiter:
        start = time.time()
        try:
            response = await client.aio.models.generate_content(
                model=model_id,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    max_output_tokens=500,
                )
            )
            latency = time.time() - start
            
            return {
                "text": response.text,
                "latency": latency,
                "prompt_tokens": response.usage_metadata.prompt_token_count,
                "output_tokens": response.usage_metadata.candidates_token_count,
            }
        except Exception as e:
            print(f"Error during generation: {e}")
            return {"text": f"Error: {e}", "latency": 0, "prompt_tokens": 0, "output_tokens": 0}

LLM Systems

Prompt Only

In [9]:
async def run_prompt_only(docs, qa_data):
    results = []
    answers = []

    async def process_item(item):
        doc_text = docs[item["document"]]
        
        prompt = f"""
Extract the exact answer span from the document.

Document:
{doc_text}

Question:
{item["question"]}

Answer:
"""

        gen_data = await generate_async(prompt)
        prediction = gen_data["text"]

        result = {
            "qa_id": item["id"],
            "system": "Prompt-Only",
            "latency": gen_data["latency"],
            "prompt_tokens_used": gen_data["prompt_tokens"],
            "output_tokens": gen_data["output_tokens"],
            "EM": exact_match(prediction, item["answer_text"])
        }

        # Prepare answer log
        answer = {
            "qa_id": item["id"],
            "question": item["question"],
            "document": item["document"],
            "prediction": prediction,
            "gold_answer": item["answer_text"]
        }
        
        return result, answer
    
    tasks = [process_item(item) for item in qa_data]
    task_outputs = await asyncio.gather(*tasks)

    for result, answer in task_outputs:
        results.append(result)
        answers.append(answer)

    with open("answers_prompt_g.json", "w") as f:
        json.dump(answers, f, indent=2)

    return results

RAG

In [10]:
def chunk_text(text, tokenizer, size=512, overlap=128):
    tokens = tokenizer.encode(text)
    chunks = []

    for i in range(0, len(tokens), size - overlap):
        chunk_tokens = tokens[i:i+size]
        chunks.append(tokenizer.decode(chunk_tokens))

    return chunks

In [11]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def build_index(docs, tokenizer):
    all_chunks = []
    metadata = []

    for name, text in docs.items():
        chunks = chunk_text(text, tokenizer)
        for chunk in chunks:
            all_chunks.append(chunk)
            metadata.append(name)

    embeddings = embedder.encode(all_chunks, convert_to_numpy=True)

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)

    return index, all_chunks

In [12]:
async def run_rag(index, chunks, qa_data):
    results = []

    answers = []

    async def process_rag_item(item):
        q_emb = embedder.encode(item["question"], convert_to_numpy=True).reshape(1, -1)
        
        start_retrieval = time.time()
        D, I = index.search(q_emb, 5)
        retrieval_time = time.time() - start_retrieval

        retrieved_chunks = [chunks[i] for i in I[0]]
        context = "\n\n".join(retrieved_chunks)

        prompt = f"""
Extract exact answer span.

Context:
{context}

Question:
{item["question"]}

Answer:
"""

        gen_data = await generate_async(prompt)
        
        prediction = gen_data["text"]

        recall = int(any(item["answer_text"].lower() in c.lower() for c in retrieved_chunks))

        result = {
            "qa_id": item["id"],
            "system": "RAG",
            "latency": gen_data["latency"] + retrieval_time,
            "prompt_tokens_used": gen_data["prompt_tokens"],
            "output_tokens": gen_data["output_tokens"],
            "EM": exact_match(prediction, item["answer_text"]),
            "Recall": recall
        }

        answer = {
            "qa_id": item["id"],
            "question": item["question"],
            "prediction": prediction,
            "gold_answer": item["answer_text"],
            "retrieved_chunks": retrieved_chunks
        }
        return result, answer

    tasks = [process_rag_item(item) for item in qa_data]
    task_outputs = await asyncio.gather(*tasks)

    for result, answer in task_outputs:
        results.append(result)
        answers.append(answer)

    with open("answers_rag_g.json", "w") as f:
        json.dump(answers, f, indent=2)

    return results

Context Optimization

In [23]:
async def run_context(docs, qa_data):
    results, answers = [], []

    async def process(item):
        doc_text = docs.get(item.get("document", ""), "")
        query = str(item.get("article", "")).lower()

        # Extract relevant context
        context = ""
        if query:
            lines = doc_text.split("\n")
            for i, line in enumerate(lines):
                if query in line.lower():
                    start = max(0, i - 10)
                    end = min(len(lines), i + 500)
                    context = "\n".join(lines[start:end])
                    break

        # Use filtered context or fallback
        base_context = context if context else doc_text

        prompt = f"""Use the provided context to answer the question.

Context:
{base_context}

Question: {item['question']}
Answer:"""

        gen_data = await generate_async(prompt)
        prediction = gen_data["text"]

        # Retry with full document if weak answer
        if ("not found" in prediction.lower() or
            "not mentioned" in prediction.lower() or
            len(prediction) < 10):

            print(f"Fallback for {item['id']}")

            full_prompt = f"""Answer using the FULL document.

Document:
{doc_text}

Question: {item['question']}
Answer:"""

            gen_data = await generate_async(full_prompt)
            prediction = gen_data["text"]

        result = {
            "qa_id": item["id"],
            "system": "Context-Filtered",
            "latency": gen_data["latency"],
            "prompt_tokens_used": gen_data["prompt_tokens"],
            "output_tokens": gen_data["output_tokens"],
            "EM": exact_match(prediction, item["answer_text"])
        }

        answer = {
            "qa_id": item["id"],
            "question": item["question"],
            "prediction": prediction,
            "gold_answer": item["answer_text"]
        }

        return result, answer

    outputs = await asyncio.gather(*(process(item) for item in qa_data))

    for r, a in outputs:
        results.append(r)
        answers.append(a)

    with open("answers_context_g.json", "w") as f:
        json.dump(answers, f, indent=2)

    return results

Run

In [24]:
docs = load_documents()
qa_data = load_all_qa()

index, chunks = build_index(docs, embedder.tokenizer)

In [13]:
print("Running Prompt-Only...")
results_prompt_G = await run_prompt_only(docs, qa_data)

with open("results_prompt_gemini.json", "w") as f:
    json.dump(results_prompt_G, f, indent=2)
print(f"Prompt-Only complete. Saved {len(results_prompt_G)} results.")

Running Prompt-Only...
Prompt-Only complete. Saved 100 results.


In [16]:
print("Running RAG...")
results_rag_G = await run_rag(index, chunks, qa_data)
with open("results_rag_gemini.json", "w") as f:
    json.dump(results_rag_G, f, indent=2)
print(f"RAG complete. Saved {len(results_rag_G)} results.")

Running RAG...
RAG complete. Saved 100 results.


In [25]:
print("Running Context-Optimized...")
results_context_G = await run_context(docs, qa_data)

with open("results_context_gemini.json", "w") as f:
    json.dump(results_context_G, f, indent=2)
print(f"Context-Optimized complete. Saved {len(results_context_G)} results.")

Running Context-Optimized...
Context-Optimized complete. Saved 100 results.


In [ ]:
all_results = results_prompt_G + results_rag_G + results_context_G